In [ ]:
# 🏥 Scenario: The Hospital’s AI Appointment Agent
# Imagine a hospital has deployed an AI-powered agent named MediLLM.
# Its job is to monitor patient appointment requests and decide whether the requested slot is available or needs rescheduling.

# Step 1 — Thinking (Reasoning)
# Whenever a patient request comes in, MediLLM first analyzes the details:
# - Example: “Request for Dr. Sharma at 10 AM tomorrow”
# - MediLLM thinks: “Analyzing appointment request: Dr. Sharma, 10 AM tomorrow.”

# Step 2 — Planning (Choosing an Action)
# Based on the thought, MediLLM decides which tool to use:
# - If the requested slot is free, it plans to confirm appointment.
# - If the slot is already booked, it plans to suggest rescheduling.

# Step 3 — Acting (Running Tools)
# MediLLM hands over the decision to AppointmentTools, which executes the action:
# - confirm_slot → returns “Appointment Confirmed”
# - suggest_reschedule → returns “Slot Unavailable, Please Reschedule”

# Step 4 — Answering (Final Decision)
# Once a clear classification is reached, MediLLM announces the decision:
# - “Decision: Appointment Confirmed”
# - “Decision: Slot Unavailable, Please Reschedule”

In [ ]:
import requests
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()
API_KEY = os.getenv("NVIDIA_API_KEY")

API_URL = "https://integrate.api.nvidia.com/v1/chat/completions"

# -------------------------------
# 🧠 LLM (Thinking)
# -------------------------------
def query_llm(prompt: str):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "meta/llama3-70b-instruct",
        "messages": [
            {
                "role": "system",
                "content": "You are a strict assistant. Give short and precise answers only."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.1
    }

    response = requests.post(API_URL, headers=headers, json=payload)
    data = response.json()

    return data["choices"][0]["message"]["content"]


# -------------------------------
# 📅 MOCK DATABASE (Booked Slots)
# -------------------------------
booked_slots = [
    {"doctor": "Dr. Sharma", "time": "10 AM tomorrow"},
    {"doctor": "Dr. Mehta", "time": "2 PM today"}
]


# -------------------------------
# 🔧 TOOLS (Acting)
# -------------------------------
def confirm_slot(request):
    return "✅ Appointment Confirmed"

def suggest_reschedule(request):
    return "Slot Unavailable, Please Reschedule"


# -------------------------------
# 🧠 HELPER (Check Availability)
# -------------------------------
def is_slot_available(doctor, time):
    for slot in booked_slots:
        if slot["doctor"].lower() == doctor.lower() and slot["time"].lower() == time.lower():
            return False
    return True


# -------------------------------
# 🤖 AGENT
# -------------------------------
def medi_agent(request: str):

    print(f"\n📄 Request: {request}")

    # Step 1: Thinking
    thought = query_llm(f"""
Extract only doctor name and time from this request.

Request: {request}

Output format:
Doctor: <name>
Time: <time>
""")
    print("🧠 Thought:", thought)

    # Extract doctor & time (simple parsing)
    doctor = "Dr. Sharma" if "sharma" in request.lower() else "Unknown"
    time = "10 AM tomorrow" if "10" in request else "Unknown"

    # Step 2: Planning
    if is_slot_available(doctor, time):
        action = confirm_slot
    else:
        action = suggest_reschedule

    print("📌 Planned Action:", action.__name__)

    # Step 3: Acting
    result = action(request)

    # Step 4: Answering
    final_answer = query_llm(f"Final decision: {result}")
    return final_answer


# -------------------------------
# 🚀 RUN
# -------------------------------
print(medi_agent("Request for Dr. Sharma at 9 AM tomorrow"))


📄 Request: Request for Dr. Sharma at 9 AM tomorrow
🧠 Thought: Doctor: Dr. Sharma
Time: 9 AM
📌 Planned Action: confirm_slot
NOTED.
